In [3]:
"""
03_hyperparameter_tuning.py -- ScoutAI Model Optimization (Dual Model)

Runs a RandomizedSearchCV over the XGBoost regressor for BOTH the 
'full' and 'performance_only' models. Calculates RMSE, R², and Adjusted R²
to evaluate model performance. Saves the new versions as '_tuned.pkl' 
only if they achieve a lower RMSE. Outputs a detailed log.
"""

import os
import sys
import pandas as pd
import numpy as np
import sqlalchemy
import joblib
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
import time
from dotenv import load_dotenv, find_dotenv

# ==========================================
# 0. CONFIG & SETUP
# ==========================================
load_dotenv(find_dotenv())

MODELS_DIR = "models"
DATA_DIR = "data"

for directory in [MODELS_DIR, DATA_DIR]:
    os.makedirs(directory, exist_ok=True)

def calculate_metrics(y_true, y_pred, n_features):
    """Calculates RMSE, R-squared, and Adjusted R-squared."""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    n_samples = len(y_true)
    # Adjusted R2 formula: 1 - [(1-R2)*(n-1)/(n-k-1)]
    adj_r2 = 1 - (1 - r2) * (n_samples - 1) / (n_samples - n_features - 1)
    return rmse, r2, adj_r2

def main():
    # ==========================================
    # 1. DATABASE CONNECTION & DATA LOAD
    # ==========================================
    db_url = os.getenv('DB_URL')
    if not db_url:
        print("[ERROR] DB_URL not found. Please ensure your .env file exists.")
        sys.exit(1)

    print("[SYSTEM] Fetching data for Hyperparameter Tuning...")
    try:
        engine = sqlalchemy.create_engine(db_url)
        df = pd.read_sql("SELECT * FROM view_scout_master", engine)
    except Exception as e:
        print(f"[ERROR] Database connection failed: {e}")
        sys.exit(1)

    df['current_market_value'] = pd.to_numeric(df['current_market_value'], errors='coerce').fillna(0)
    df = df[df['current_market_value'] > 0].copy()
    y = np.log1p(df['current_market_value'])

    # ==========================================
    # 2. FEATURE ENGINEERING
    # ==========================================
    print("[SYSTEM] Engineering features...")
    cols_to_clean = [
        'age', 'total_appearances', 'international_caps', 'international_goals',
        'max_career_transfer_fee', 'most_recent_transfer_fee', 'height_in_cm',
        'minutes_per_match', 'total_goals', 'total_assists', 'goals_per_90',
        'assists_per_90', 'total_yellow_cards', 'total_red_cards', 'club_squad_size',
        'club_avg_age', 'stadium_seats', 'foreigners_percentage', 'contract_months_remaining',
    ]
    for col in cols_to_clean:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    df['wonderkid_hype'] = np.where(df['age'] <= 22, (df['total_appearances'] + (df['international_caps'] * 3)), 0)
    
    conditions = [
        (df['passport_power_rank'].fillna(999) <= 15), 
        (df['passport_power_rank'].fillna(999) > 15) & (df['passport_power_rank'].fillna(999) <= 60)
    ]
    df['passport_tier'] = np.select(conditions, ['Tier_1', 'Tier_2'], default='Tier_3')
    
    league_weights = {
        'Premier League': 1.5, 'LaLiga': 1.4, 'Serie A': 1.3, 'Bundesliga': 1.3, 
        'Ligue 1': 1.2, 'Eredivisie': 1.15, 'Liga Portugal': 1.15, 'Süper Lig': 1.05
    }
    df['league_coefficient'] = df['competition_name'].map(league_weights).fillna(1.0)
    df['detailed_position'] = df['sub_position'].fillna(df['position_group']).astype(str)

    FULL_FEATURES = [
        'age', 'height_in_cm', 'total_appearances', 'minutes_per_match', 'total_goals', 
        'total_assists', 'goals_per_90', 'assists_per_90', 'total_yellow_cards', 'total_red_cards', 
        'international_caps', 'international_goals', 'club_squad_size', 'club_avg_age', 
        'stadium_seats', 'foreigners_percentage', 'contract_months_remaining', 'wonderkid_hype', 
        'league_coefficient', 'has_transfer_history', 'max_career_transfer_fee', 
        'most_recent_transfer_fee', 'detailed_position', 'foot', 'passport_tier'
    ]
    
    PERFORMANCE_FEATURES = [
        'age', 'height_in_cm', 'total_appearances', 'minutes_per_match', 'total_goals', 
        'total_assists', 'goals_per_90', 'assists_per_90', 'total_yellow_cards', 'total_red_cards', 
        'international_caps', 'international_goals', 'club_squad_size', 'club_avg_age', 
        'stadium_seats', 'foreigners_percentage', 'wonderkid_hype', 'league_coefficient', 
        'detailed_position', 'foot', 'passport_tier'
    ]

    models_to_tune = [
        {"label": "full", "features": FULL_FEATURES}, 
        {"label": "performance_only", "features": PERFORMANCE_FEATURES}
    ]

    report_lines = []
    header = (
        "==========================================\n"
        " 🛠️ SCOUT AI: DUAL-MODEL OPTIMIZATION 🛠️\n"
        "==========================================\n"
    )
    print(header, end="")
    report_lines.append(header)

    # ==========================================
    # 3. HYPERPARAMETER TUNING LOOP
    # ==========================================
    for model_info in models_to_tune:
        model_label = model_info["label"]
        features = model_info["features"]
        
        print(f"\n[SYSTEM] Starting optimization for '{model_label}' model...")

        X = df[features]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        n_features = X_train.shape[1]

        model_path = os.path.join(MODELS_DIR, f'scout_model_{model_label}.pkl')
        try:
            old_pipeline = joblib.load(model_path)
        except FileNotFoundError:
            print(f"[ERROR] Could not find '{model_path}'. Skipping {model_label} tuning.")
            continue

        old_preds = old_pipeline.predict(X_test)
        old_rmse, old_r2, old_adj_r2 = calculate_metrics(np.expm1(y_test), np.expm1(old_preds), n_features)

        param_distributions = {
            'regressor__n_estimators': [400, 800, 1200, 1600],
            'regressor__max_depth': [4, 5, 6, 7, 8],
            'regressor__learning_rate': [0.01, 0.02, 0.03, 0.05, 0.08],
            'regressor__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
            'regressor__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
            'regressor__min_child_weight': [1, 3, 5, 7],
            'regressor__gamma': [0, 0.1, 0.3, 0.5],
            'regressor__reg_alpha': [0, 0.1, 1],
            'regressor__reg_lambda': [1, 1.5, 2],
        }

        tuning_pipeline = Pipeline(steps=[
            ('preprocessor', old_pipeline.named_steps['preprocessor']), 
            ('regressor', old_pipeline.named_steps['regressor'])
        ])
        
        print(f"[SYSTEM] Running 5-Fold Cross-Validation for {model_label}...")
        start_time = time.time()
        
        random_search = RandomizedSearchCV(
            tuning_pipeline, 
            param_distributions, 
            n_iter=25, 
            cv=5, 
            scoring='neg_mean_squared_error', 
            random_state=42, 
            n_jobs=1, 
            verbose=1
        )
        random_search.fit(X_train, y_train)

        elapsed_time = time.time() - start_time
        mean_cv_score = np.sqrt(-random_search.best_score_)

        best_pipeline = random_search.best_estimator_
        new_preds = best_pipeline.predict(X_test)
        new_rmse, new_r2, new_adj_r2 = calculate_metrics(np.expm1(y_test), np.expm1(new_preds), n_features)

        # ==========================================
        # 4. REPORTING
        # ==========================================
        report_block = f"\n--- {model_label.upper()} MODEL ---\n\n"
        
        report_block += "Best Parameters:\n----------------\n"
        for key, value in random_search.best_params_.items():
            clean_key = key.replace("regressor__", "")
            report_block += f"  {clean_key}: {value}\n"
            
        report_block += f"\nTraining Time:\n--------------\n  {elapsed_time:.1f} sec\n\n"
        
        report_block += "Metrics:\n--------\n"
        report_block += f"  Old RMSE      : €{old_rmse:,.0f}\n"
        report_block += f"  New RMSE      : €{new_rmse:,.0f}\n\n"
        report_block += f"  Old R²        : {old_r2:.4f}\n"
        report_block += f"  New R²        : {new_r2:.4f}\n\n"
        report_block += f"  Old Adj R²    : {old_adj_r2:.4f}\n"
        report_block += f"  New Adj R²    : {new_adj_r2:.4f}\n\n"
        
        report_block += "Decision:\n---------\n"
        if new_rmse < old_rmse:
            rmse_diff = old_rmse - new_rmse
            report_block += f"  ✅ SUCCESS: The tuned model is better!\n"
            report_block += f"     Improvement: €{rmse_diff:,.0f} reduction in RMSE.\n"
            report_block += f"     5-Fold CV Log-RMSE (Best): {mean_cv_score:.4f}\n"
            report_block += f"     Saving as 'scout_model_{model_label}_tuned.pkl'...\n"
            new_model_path = os.path.join(MODELS_DIR, f'scout_model_{model_label}_tuned.pkl')
            joblib.dump(best_pipeline, new_model_path)
        else:
            report_block += f"  ⚠️ NOTE: The old model performed at least as well.\n"

        print(report_block, end="")
        report_lines.append(report_block)

    # Save report to TXT
    txt_export_path = os.path.join(DATA_DIR, 'tuning_results_log.txt')
    with open(txt_export_path, 'w', encoding='utf-8') as f:
        f.writelines(report_lines)
    
    print(f"\n[SYSTEM] Tuning results securely saved to '{txt_export_path}'")

if __name__ == "__main__":
    main()

[SYSTEM] Fetching data for Hyperparameter Tuning...
[SYSTEM] Engineering features...
 🛠️ SCOUT AI: DUAL-MODEL OPTIMIZATION 🛠️

[SYSTEM] Starting optimization for 'full' model...
[SYSTEM] Running 5-Fold Cross-Validation for full...
Fitting 5 folds for each of 25 candidates, totalling 125 fits

--- FULL MODEL ---

Best Parameters:
----------------
  subsample: 0.8
  reg_lambda: 1.5
  reg_alpha: 1
  n_estimators: 1600
  min_child_weight: 1
  max_depth: 7
  learning_rate: 0.02
  gamma: 0.5
  colsample_bytree: 0.7

Training Time:
--------------
  435.3 sec

Metrics:
--------
  Old RMSE      : €4,022,706
  New RMSE      : €3,848,864

  Old R²        : 0.6992
  New R²        : 0.7246

  Old Adj R²    : 0.6973
  New Adj R²    : 0.7229

Decision:
---------
  ✅ SUCCESS: The tuned model is better!
     Improvement: €173,842 reduction in RMSE.
     5-Fold CV Log-RMSE (Best): 0.7921
     Saving as 'scout_model_full_tuned.pkl'...

[SYSTEM] Starting optimization for 'performance_only' model...
[SYSTE